# Dunnhumby M1 vs preference-anchored M2

Dunnhumby 전체 학습기간, seed 42, validation만 사용해 외부 `M1@64`와 `joint_nv_anchored` 두 모형만 비교합니다. H&M, 대조군, train-internal 변수 타당성 재계산, test, holdout은 실행하지 않습니다.

M2는 `ID|N|V` layer-0 표현을 하나의 binary LightGCN으로 공동 전파합니다. 학습목적은 `0.5*BPR(S_full) + 0.5*BPR(S_ID)`이며 표본별 CLV 가중치는 사용하지 않습니다. 사용자 N/V 관측이 유효하지 않으면 해당 MLP 출력 블록을 정확히 0으로 마스킹합니다. epoch별 진행상태는 Drive에 저장되어 같은 코드와 설정으로 다시 실행하면 이어서 학습합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, subprocess

REVIEWED_SHA = 'e3865fafdffd02c04409c3eaed20d1743c159d33'
repo = Path('/content/clv-m2-lightgcn-runner')
os.chdir('/content')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run([
    'git', 'clone', '-q',
    'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)
], check=True)
os.chdir(repo)
subprocess.run(['git', 'checkout', '-q', REVIEWED_SHA], check=True)
assert subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], text=True
).strip() == REVIEWED_SHA
print('code:', REVIEWED_SHA)

In [ ]:
import importlib, json, sys, torch
for module_name, module in list(sys.modules.items()):
    module_path = getattr(module, '__file__', '') or ''
    if module_path and str(repo) in str(module_path):
        del sys.modules[module_name]
importlib.invalidate_caches()

from IPython.display import display
import pandas as pd
from lightgcn_clv_joint_nv import (
    configure_anchored_dunnhumby_run,
    preflight_summary,
    run_experiment,
)
assert torch.cuda.is_available(), (
    '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
cfg = configure_anchored_dunnhumby_run()
summary = preflight_summary(cfg)
print(json.dumps(summary, ensure_ascii=False, indent=2))
assert summary['dataset'] == 'dunnhumby'
assert summary['seed'] == 42
assert summary['models'] == ['m1', 'joint_nv_anchored']
assert summary['eval_test'] is False
assert summary['eval_holdout'] is False
assert summary['compute_variable_validity'] is False
print('설정 확인 완료. 다음 셀은 바로 학습을 시작합니다.')

In [ ]:
result_df = run_experiment(cfg)

In [ ]:
columns = [
    'model_id', 'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50', 'revenue@10', 'revenue@20', 'revenue@50',
    'arp@10', 'coverage@10', 'n_distinct@10',
    'exposure_entropy@10', 'eff_catalog@10',
    'top10_share@10', 'top100_share@10', 'value_alignment',
]
absolute = result_df[[c for c in columns if c in result_df.columns]].copy()
print('===== 절대지표 =====')
display(absolute)

m1 = result_df.set_index('model_id').loc['m1']
m2 = result_df.set_index('model_id').loc['joint_nv_anchored']
metric_names = [c for c in columns[1:] if c in result_df.columns]
comparison = pd.DataFrame({
    'metric': metric_names,
    'M1': [m1[c] for c in metric_names],
    'anchored_M2': [m2[c] for c in metric_names],
})
comparison['absolute_delta'] = comparison['anchored_M2'] - comparison['M1']
comparison['relative_change_pct'] = (
    comparison['absolute_delta'] / comparison['M1'].replace(0, float('nan')) * 100
)
print('===== M1 대비 변화 =====')
display(comparison)

print('===== paired delta =====')
display(pd.read_csv(result_df.attrs['result_paths']['delta_csv']))
print('사용자 N/V 유효 마스크:', result_df.attrs['user_axis_mask_summary'])
print('판정:', result_df.attrs['decision'])
print('결과 파일:', result_df.attrs['result_paths'])
print('완료. 위 절대지표와 M1 대비 변화표를 그대로 공유해 주세요.')